[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/49_dpo_loss_solution.ipynb)

# 🔴 Solution: DPO (Direct Preference Optimization) Loss

*RLHF & Preference Losses · Hard*

Reference implementation. Try it yourself in `49_dpo_loss.ipynb` first.

---
Implement the **DPO** loss.

$$\mathcal{L} = -\log\sigma\!\left(\beta\Big[
\big(\log\pi_\theta(y_w|x) - \log\pi_{\text{ref}}(y_w|x)\big) -
\big(\log\pi_\theta(y_l|x) - \log\pi_{\text{ref}}(y_l|x)\big)
\Big]\right)$$

where $y_w$ is the **chosen** (preferred) completion and $y_l$ the **rejected**
one. All four inputs are already-summed sequence log-probabilities of shape
`(batch,)`.

### Signature
```python
def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    ...  # -> scalar, mean over the batch
```

### Rules
- Return the **mean** over the batch, as a scalar
- Use a numerically stable log-sigmoid — `jnp.log(sigmoid(x))` is not acceptable
- Do not detach or stop-gradient the policy terms; the reference terms are
  plain constants here (already computed under no-grad upstream)

### What the reference model is doing there
Without $\pi_{\text{ref}}$ the objective would happily drive
$\log\pi_\theta(y_w)$ to zero and $\log\pi_\theta(y_l)$ to $-\infty$ — perfect
preference accuracy, destroyed model. Subtracting the reference log-probs turns
the quantity being ranked into a *log-ratio*, and that ratio is exactly the
implicit reward of a KL-constrained RLHF problem,

$$r(x,y) = \beta\log\frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)} + \beta\log Z(x)$$

where the intractable $\log Z(x)$ depends only on the prompt — so it cancels the
moment you subtract two completions of the *same* prompt, which is why the loss
never has to compute it. The KL penalty is therefore not an extra term bolted
on; it is baked into the parameterisation. $\beta$ is the KL strength: large
$\beta$ keeps you near the reference, small $\beta$ lets you drift.

### Why this replaced PPO-style RLHF
Classic RLHF needs a separately-trained reward model plus an online RL loop with
a value network ([[ppo_loss]]). DPO's insight is that for the KL-constrained
objective the optimal policy has a closed form, which can be inverted to express
the reward *in terms of the policy itself* — so the reward model cancels out and
what remains is a binary classification loss on offline preference pairs. No
sampling, no value network, no reward model.

### The trap
At initialisation $\pi_\theta = \pi_{\text{ref}}$, so every margin is exactly 0
and the loss is $-\log\sigma(0) = \log 2 \approx 0.6931$. If your first training
step does not report ~0.693, something is wrong — that constant is the standard
sanity check, and it is what the tests pin down.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    # Implicit rewards, up to the shared beta factor.
    chosen_ratio = policy_chosen_logps - ref_chosen_logps
    rejected_ratio = policy_rejected_logps - ref_rejected_logps

    logits = beta * (chosen_ratio - rejected_ratio)

    # log sigmoid(x) == -softplus(-x); stable for large |x| in both directions.
    return -jnp.mean(jax.nn.log_sigmoid(logits))

In [ ]:
# 🔍 Verify
import jax.numpy as jnp

# At init the policy IS the reference -> every margin is 0 -> loss = log 2.
z = jnp.zeros(4)
print("at init:      ", float(dpo_loss(z, z, z, z)))
print("log 2 =       ", float(jnp.log(2.0)))

# Policy prefers the chosen completion more than the reference does -> lower loss.
pc = jnp.array([0.5, 0.5, 0.5, 0.5])
print("chosen boosted:", float(dpo_loss(pc, z, z, z)))

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("dpo_loss")